# Stage 2a: intraday reaction and drift around CPI releases

Uses `futures_1m` (Stage 2 pull) and Stage 1 results. **Holdout releases (last 12 months) are excluded everywhere.**

Timing (no look-ahead): Databento 1-minute bars are stamped at their start, so the price *at* time t is the close of the bar that started one minute earlier. "Pre" is the last trade before 8:30:00.

Sections: (1) reaction to the surprise at 8:31 and 8:35; (2) does the move continue or reverse later in the day, and does the drift load on the Kalshi surprise or the trend deviation; (3) a walk-forward drift rule with costs, written in the sleeve interface (signal table + per-event net P&L).

In [ ]:
MOUNT_DRIVE = True
NW = None                     # events are independent: heteroskedasticity-robust (HC1) standard errors
MIN_TRAIN = 15                # events before the first walk-forward trade
ENTRY, EXIT = "08:35", "16:00"
# round-trip cost per contract, in price points: one tick of spread + ~$5 commissions/fees. ASSUMPTIONS: check against your broker.
COST = {"ZT.v.0": 1/256 + 5/2000, "ZN.v.0": 1/64 + 5/1000, "SR3.c.1": 0.0025 + 5/2500, "SR3.c.2": 0.0025 + 5/2500,
        "SR3.c.3": 0.005 + 5/2500, "SR3.c.4": 0.005 + 5/2500, "ES.v.0": 0.25 + 5/50, "6E.v.0": 0.00005 + 5/125000}
def cost_of(sym):
    if sym in COST: return COST[sym]
    if sym.startswith("SR3"): return 0.0025 + 5/2500
    raise KeyError(sym)
# convert price changes to "hawkish" units: + = higher yields / rates / dollar, lower stocks
DUR = {"ZT.v.0": 1.9, "ZN.v.0": 6.5}   # approximate modified durations (assumption)

import numpy as np, pandas as pd, statsmodels.api as sm
from pathlib import Path
from scipy import stats
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW = Path("/content/drive/MyDrive/macro-news-pricing/data/raw")
else:
    RAW = Path("/content/data/raw")
F = pd.read_parquet(RAW / "futures_1m.parquet")
W = pd.read_parquet(RAW / "stage2_event_windows.parquet")
h = pd.read_parquet(RAW / "stage1_results_cpi.parquet")
c = pd.read_parquet(RAW / "stage1_results_core_cpi.parquet")
act = pd.read_parquet(RAW / "nowcast_actuals.parquet")
F["start_et"] = pd.to_datetime(F.ts_event, utc=True).dt.tz_convert("America/New_York")
F["release_date"] = pd.to_datetime(F.release_date)
print(F.groupby("symbol").release_date.nunique().to_string())

## Surprises (from Stage 1) and the trend deviation

In [ ]:
MON = {m: i for i, m in enumerate("JAN FEB MAR APR MAY JUN JUL AUG SEP OCT NOV DEC".split(), 1)}
tm = lambda t: f"20{t.split('-')[1][:2]}-{MON[t.split('-')[1][2:5]]:02d}"
S = h[["release_date", "event_ticker", "y", "mean_raw", "mean_corrected", "nowcast_point"]].rename(
    columns={"y": "y_h", "mean_raw": "k_h", "mean_corrected": "kc_h", "nowcast_point": "n_h"})
S = S.merge(c[["release_date", "y", "mean_raw", "nowcast_point"]].rename(
    columns={"y": "y_c", "mean_raw": "k_c", "nowcast_point": "n_c"}), on="release_date", how="left")
S["s_head"], S["s_core"] = S.y_h - S.k_h, S.y_c - S.k_c          # surprise vs raw Kalshi mean (correction did not transfer)
core_act = act[act.measure == "core_cpi"].sort_values("target_month").set_index("target_month").value
trend = core_act.shift(1).rolling(12).mean()
S["s_trend"] = S.y_c - S.event_ticker.map(tm).map(trend)
S["release_date"] = pd.to_datetime(S.release_date)
S = S.merge(W[["release_date", "holdout"]], on="release_date", how="inner")
print(f"{len(S)} release days; holdout excluded from here on: {int(S.holdout.sum())}")
S = S[~S.holdout].copy()

## Prices at fixed times and returns in hawkish units

In [ ]:
ANCH = {"pre": (0, "08:30"), "t0831": (0, "08:31"), "t0835": (0, "08:35"), "t0845": (0, "08:45"),
        "t0930": (0, "09:30"), "t1200": (0, "12:00"), "t1600": (0, "16:00"), "prev1605": (-1, "16:05")}

def price_at(g, day, hhmm, back_min=30):
    """close of the last bar that ENDED by hh:mm (bar start <= t - 1 min), within back_min minutes"""
    t = day.tz_localize("America/New_York") + pd.Timedelta(hours=int(hhmm[:2]), minutes=int(hhmm[3:]))
    x = g[(g.start_et <= t - pd.Timedelta(minutes=1)) & (g.start_et > t - pd.Timedelta(minutes=back_min + 1))]
    return (x.close.iloc[-1], x.instrument_id.iloc[-1]) if len(x) else (np.nan, np.nan)

rows = []
for (sym, d), g in F[F.release_date.isin(S.release_date)].groupby(["symbol", "release_date"]):
    r = {"symbol": sym, "release_date": d}
    for k, (off, hhmm) in ANCH.items():
        day = d - pd.offsets.BDay(1) if off == -1 else d
        r[k], r[k + "_id"] = price_at(g, day, hhmm, back_min=600 if k == "prev1605" else 30)
    rows.append(r)
P = pd.DataFrame(rows)

def hawk(sym, p1, p0):
    """price change -> hawkish units: bp of yield/rate for rates, % (sign-flipped for stocks and euro) otherwise"""
    if sym in DUR:  return -(p1 - p0) / p0 / DUR[sym] * 1e4
    if sym.startswith("SR3"): return -(p1 - p0) * 100
    return -(p1 / p0 - 1) * 100       # ES, 6E: falling stocks / falling euro = hawkish

def ret(p, a, b):
    same = p[a + "_id"] == p[b + "_id"]           # drop windows that straddle a contract roll
    return np.where(same, [hawk(s, y, x) for s, y, x in zip(p.symbol, p[b], p[a])], np.nan)

WIN = {"jump_1m": ("pre", "t0831"), "jump_5m": ("pre", "t0835"), "d_0835_0930": ("t0835", "t0930"),
       "d_0930_1600": ("t0930", "t1600"), "d_0835_1600": ("t0835", "t1600"), "day": ("prev1605", "t1600"), "pre_move": ("prev1605", "pre")}
for k, (a, b) in WIN.items():
    P[k] = ret(P, a, b)
P = P.merge(S[["release_date", "s_head", "s_core", "s_trend"]], on="release_date")
P.to_parquet(RAW / "stage2a_event_returns.parquet", index=False)
print(P.groupby("symbol")[list(WIN)].agg(lambda x: x.abs().median()).round(2).to_string())
print("(median absolute move; bp for ZT/ZN/SR3, % for ES/6E)")

## 1. Reaction to the surprise

In [ ]:
def ols(df, y, xs):
    d = df[[y] + xs].dropna()
    if len(d) < 10: return None
    return sm.OLS(d[y], sm.add_constant(d[xs])).fit(cov_type="HC1")
pd.set_option("display.width", 250)
out = []
for sym, g in P.groupby("symbol"):
    for y in ("jump_1m", "jump_5m", "day"):
        for xs in (["s_head", "s_core"], ["s_head", "s_core", "s_trend"]):
            m = ols(g, y, xs)
            if m is None: continue
            row = {"symbol": sym, "y": y, "x": "+".join(xs), "n": int(m.nobs), "R2": m.rsquared}
            for x in xs: row[f"b_{x}"], row[f"p_{x}"] = m.params[x], m.pvalues[x]
            out.append(row)
print(pd.DataFrame(out).round(3).to_string(index=False))

## 2. Continuation or reversal after the jump

In [ ]:
out = []
for sym, g in P.groupby("symbol"):
    for y in ("d_0835_0930", "d_0930_1600", "d_0835_1600"):
        for xs in (["jump_5m"], ["s_head", "s_core", "s_trend"], ["jump_5m", "s_trend"]):
            m = ols(g, y, xs)
            if m is None: continue
            row = {"symbol": sym, "y": y, "x": "+".join(xs), "n": int(m.nobs), "R2": m.rsquared}
            for x in xs: row[f"b_{x}"], row[f"p_{x}"] = m.params[x], m.pvalues[x]
            out.append(row)
print("b_jump_5m > 0 = the move continues; < 0 = it reverses")
print(pd.DataFrame(out).round(3).to_string(index=False))

## 3. Walk-forward drift rule, net of costs (sleeve interface)
For each event after the first `MIN_TRAIN`: fit `drift(ENTRY->EXIT) ~ jump_5m + s_trend` on earlier events only, predict this event's drift at 8:35 (everything used is known by then), go long or short one contract in the predicted direction, exit at 16:00. P&L in points per contract, net of the round-trip cost. One rule, fixed before results; no tuning.

In [ ]:
def raw_pts(p, a, b):
    same = p[a + "_id"] == p[b + "_id"]
    return np.where(same, p[b] - p[a], np.nan)
P["pts_entry_exit"] = raw_pts(P, "t0835", "t1600")
sig = []
for sym, g in P.sort_values("release_date").groupby("symbol"):
    g = g.dropna(subset=["jump_5m", "s_trend", "d_0835_1600", "pts_entry_exit"]).reset_index(drop=True)
    for i in range(MIN_TRAIN, len(g)):
        tr, r = g.iloc[:i], g.iloc[i]
        m = sm.OLS(tr.d_0835_1600, sm.add_constant(tr[["jump_5m", "s_trend"]])).fit()
        pred = float(m.params["const"] + m.params["jump_5m"] * r.jump_5m + m.params["s_trend"] * r.s_trend)
        # hawkish units are minus price for rates/ES/6E: hawkish prediction -> short the contract
        pos = -np.sign(pred)
        gross = pos * r.pts_entry_exit
        entry = r.release_date.tz_localize("America/New_York") + pd.Timedelta(hours=8, minutes=35)
        sig.append({"timestamp": entry, "instrument": sym, "signal": pred, "info_time": entry, "position": pos,
                    "pnl_gross_pts": gross, "pnl_net_pts": gross - cost_of(sym) * abs(pos), "cost_pts": cost_of(sym)})
SIG = pd.DataFrame(sig)
SIG.to_parquet(RAW / "stage2a_signals.parquet", index=False)
summ = SIG.groupby("instrument").apply(lambda d: pd.Series({
    "trades": len(d), "hit_rate": (d.pnl_gross_pts > 0).mean(),
    "mean_gross": d.pnl_gross_pts.mean(), "mean_net": d.pnl_net_pts.mean(), "cost": d.cost_pts.iloc[0],
    "t_net": d.pnl_net_pts.mean() / (d.pnl_net_pts.std(ddof=1) / np.sqrt(len(d))),
    "sharpe_per_event": d.pnl_net_pts.mean() / d.pnl_net_pts.std(ddof=1)}), include_groups=False)
print("Walk-forward rule, points per contract per event (ZT 1pt = $2000, ZN $1000, SR3 $2500, ES $50, 6E $125,000):")
print(summ.round(4).to_string())
print("\nAnnualized Sharpe if traded on 12 CPI days a year = sharpe_per_event x sqrt(12); read with the trade count in mind.")

## 4. Where does the trend-deviation effect happen? And an ES check suggested by section 2
(a) Splits the release-day move into before 8:30, the 5-minute jump and the drift to 16:00, and shows the trend-deviation coefficient in each piece. The pieces add up to the daily move (up to the 8:30 to 8:35 overlap and dropped rolls).
(b) ES rule with no fitted parameters: at 8:35 trade ES in the direction of the 5-minute jump, exit at 9:30 (cash open), net of costs. Section 2 suggested this pattern on the same events, so this is in-sample; only the holdout can confirm it. Do not tune it.


In [ ]:
xs = ["s_head", "s_core", "s_trend"]
rows = []
for sym in ("ZT.v.0", "ZN.v.0", "ES.v.0", "6E.v.0"):
    g = P[P.symbol == sym].dropna(subset=["pre_move", "jump_5m", "d_0835_1600", "day"] + xs)
    for y in ("pre_move", "jump_5m", "d_0835_1600", "day"):
        m = sm.OLS(g[y], sm.add_constant(g[xs])).fit(cov_type="HC1")
        rows.append({"symbol": sym, "piece": y, "n": int(m.nobs), "b_s_trend": m.params["s_trend"], "p_s_trend": m.pvalues["s_trend"],
                     "b_s_head": m.params["s_head"], "b_s_core": m.params["s_core"], "R2": m.rsquared})
print("Same events in every row of a symbol, so the pieces' coefficients roughly add up to the 'day' row:")
print(pd.DataFrame(rows).round(3).to_string(index=False))

E = P[P.symbol == "ES.v.0"].copy()
E["pts"] = raw_pts(E, "t0835", "t0930")
E = E.dropna(subset=["jump_5m", "pts"])
pos = -np.sign(E.jump_5m)                      # hawkish jump = ES fell -> keep short; dovish -> long
net = pos * E.pts - cost_of("ES.v.0") * pos.abs()
print(f"\nES jump-continuation 8:35->9:30 (in-sample, {len(E)} events): hit {(pos * E.pts > 0).mean():.2f}, "
      f"mean gross {np.mean(pos * E.pts):.2f} pts, mean net {net.mean():.2f} pts (${50 * net.mean():.0f}/contract), "
      f"t {net.mean() / (net.std(ddof=1) / np.sqrt(len(net))):.2f}")


## 5. Confound check: is the 'trend' effect a regime, or a pre-release signal?
Identity: `s_trend = s_core + e_trend`, where `e_trend = Kalshi core mean - trailing 12-month core` is known at 16:00 the day before. So with `s_core` in the regression, the 'trend' coefficient is really the coefficient on `e_trend`, an ex-ante quantity. Below: the same regressions written with `e_trend`, then with year fixed effects, then also controlling for the inflation level (the trailing average). If the effect survives the fixed effects, it is not just 2022-23 drifting yields.
Then a sign rule with nothing fitted: position by sign(e_trend) at 16:05 the day before, held to 8:30 (pre-release) or to 9:30, net of costs. In-sample: the pattern was found on these events.


In [ ]:
import warnings; warnings.filterwarnings("ignore", category=UserWarning)
Q = P.copy()
Q["e_trend"] = Q.s_trend - Q.s_core                     # = Kalshi core mean - trailing 12m core, known pre-release
Q["infl_level"] = Q.release_date.map(S.set_index("release_date").pipe(lambda s: s.y_c - s.s_trend))   # trailing 12m core
Q["year"] = Q.release_date.dt.year.astype(str)
pieces = ["pre_move", "jump_5m", "d_0835_0930", "d_0835_1600"]
specs = {"base": ["s_head", "s_core", "e_trend"],
         "+ year FE": ["s_head", "s_core", "e_trend", "FE"],
         "+ year FE + infl level": ["s_head", "s_core", "e_trend", "infl_level", "FE"]}
rows = []
for sym in ("ZT.v.0", "ZN.v.0", "SR3.v.0", "SR3.v.1", "SR3.v.2", "SR3.v.3", "ES.v.0", "6E.v.0"):
    g = Q[Q.symbol == sym]
    for y in pieces:
        for lab, xs in specs.items():
            cols = [x for x in xs if x != "FE"]
            d = g.dropna(subset=[y] + cols)
            if len(d) < 15: continue
            X = d[cols]
            if "FE" in xs:
                X = pd.concat([X, pd.get_dummies(d.year, prefix="y", drop_first=True, dtype=float)], axis=1)
            m = sm.OLS(d[y], sm.add_constant(X)).fit(cov_type="HC1")
            rows.append({"symbol": sym, "piece": y, "spec": lab, "n": int(m.nobs),
                         "b_e_trend": m.params["e_trend"], "p_e_trend": m.pvalues["e_trend"], "R2": m.rsquared})
T = pd.DataFrame(rows)
print(T.pivot_table(index=["symbol", "piece"], columns="spec", values=["b_e_trend", "p_e_trend"])
      .reindex(columns=list(specs), level=1).round(3).to_string())

print("\nSign rule on e_trend, nothing fitted (in-sample). Hawkish e_trend -> short the contract (long ES/6E in hawkish units = short price).")
out = []
for sym in ("ZT.v.0", "ZN.v.0", "SR3.v.1", "SR3.v.3"):
    g = Q[Q.symbol == sym].dropna(subset=["e_trend"])
    for lab, (a, b) in {"16:05 -> 8:30": ("prev1605", "pre"), "8:35 -> 9:30": ("t0835", "t0930"), "16:05 -> 9:30": ("prev1605", "t0930")}.items():
        pts = pd.Series(raw_pts(g, a, b), index=g.index)
        ok = pts.notna() & (g.e_trend != 0)
        pos = -np.sign(g.e_trend[ok])
        net = pos * pts[ok] - cost_of(sym)
        out.append({"symbol": sym, "window": lab, "n": int(ok.sum()), "hit": (pos * pts[ok] > 0).mean(),
                    "mean_net_pts": net.mean(), "t_net": net.mean() / (net.std(ddof=1) / np.sqrt(len(net))) if len(net) > 2 else np.nan})
print(pd.DataFrame(out).round(4).to_string(index=False))
